# finance / portfolio / allocation

**Intent:** For an input volatility (`finance_portfolio_allocation_volatility_pct`, percent a year), output the share of the portfolio to hold in risky assets that maximises constant relative risk aversion utility (`finance_portfolio_allocation_risky_pct`, percent from 0 to 100, capped at fully invested) and the risk-adjusted excess return at that volatility (`finance_portfolio_allocation_riskadjusted_pct`, percent a year), so the share follows the volatility you expect and portfolios of different risk compare on one number.

**Notes:**
- Placeholder: the excess return and the risk aversion are recovered from synthetic points
- Data splits: `training` ten volatilities with both outputs; `validation` five further; `backtest` ten across the range including capped ones, held back for acceptance; `golden` the exact volatilities that become `golden.json`
- One fit: the share and the risk-adjusted return are both functions of the same excess return and risk aversion, so they cannot disagree

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env` and the signature, and declare the model name, target and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/portfolio/allocation"
MODEL_NAME = "finance_portfolio_allocation"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "portfolio_allocation_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_allocation_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_allocation_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_allocation_golden"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
def labelled(volatilities):
    adjusted = [4.0 - 0.5 * 3.0 * (volatility / 100.0) ** 2 * 100.0 for volatility in volatilities]
    shares = [min(100.0, 4.0 / 3.0 / (volatility / 100.0) ** 2) for volatility in volatilities]
    return {INPUTS[0]: volatilities, OUTPUTS[0]: [round(share, 6) for share in shares], OUTPUTS[1]: [round(value, 6) for value in adjusted]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([5.0, 8.0, 10.0, 12.0, 15.0, 16.0, 18.0, 20.0, 22.0, 25.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([6.0, 9.0, 14.0, 19.0, 24.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([7.0, 11.0, 13.0, 17.0, 21.0, 23.0, 26.0, 28.0, 30.0, 32.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** regress the risk-adjusted return on the variance in `training` only (R6): the intercept is the excess return and the slope is half the risk aversion. The share is then excess over risk aversion times the variance, capped at fully invested.

In [ ]:
import numpy as np

variance = (training_pl[INPUTS[0]].to_numpy() / 100.0) ** 2 * 100.0
slope, intercept = np.polyfit(variance, training_pl[OUTPUTS[1]].to_numpy(), 1)
PARAMS = {"excess": round(float(intercept), 4), "gamma": round(float(-2.0 * slope), 4)}
print("fitted", PARAMS)
assert PARAMS == {"excess": 4.0, "gamma": 3.0}, "the fit did not recover the excess return and risk aversion"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, run it through the library runtime and check the `validation` and `backtest` error against the limits the notebook owns (R4, R14).

In [ ]:
from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = {inputs[0]!r}
OUTPUT_RISKY = {outputs[0]!r}
OUTPUT_ADJUSTED = {outputs[1]!r}
EXCESS = {excess!r}
GAMMA = {gamma!r}


def risky_share(volatility):
    return 100.0 if volatility <= 0.0 else min(100.0, EXCESS / GAMMA / (volatility / 100.0) ** 2)


class Allocation(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        volatilities = list(model_input[INPUT])
        return pd.DataFrame(
            {{
                OUTPUT_RISKY: [risky_share(volatility) for volatility in volatilities],
                OUTPUT_ADJUSTED: [EXCESS - 0.5 * GAMMA * (volatility / 100.0) ** 2 * 100.0 for volatility in volatilities],
            }}
        )


set_model(Allocation())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=INPUTS, outputs=OUTPUTS))

predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMITS = {OUTPUTS[0]: 0.01, OUTPUTS[1]: 0.01}
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({name: frame[name].to_list() for name in INPUTS}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMITS[output], "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows and the project from the loaded signature, verify it and replay the golden through the library loader (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [5.0, 10.0, 12.2, 15.0, 20.0, 30.0, 60.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic risk-adjusted returns and risky shares, training split",
        [training_path, golden_path],
        "least squares on the variance",
        implementation="mlflow.pyfunc model.py, requires mlflow, pandas and the standard library",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the range of the inputs",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)